In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

Pré-traitement des données

In [ ]:
# Chargement des données
df = pd.read_csv('/content/Loan_Data.csv')

In [ ]:
df

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0
...,...,...,...,...,...,...,...,...
9995,3972488,0,3033.647103,2553.733144,42691.62787,5,697,0
9996,6184073,1,4146.239304,5458.163525,79969.50521,8,615,0
9997,6694516,2,3088.223727,4813.090925,38192.67591,5,596,0
9998,3942961,0,3288.901666,1043.099660,50929.37206,2,647,0


In [ ]:
# Features & target
X = df.drop('default', axis=1)
y = df['default']
df

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0
...,...,...,...,...,...,...,...,...
9995,3972488,0,3033.647103,2553.733144,42691.62787,5,697,0
9996,6184073,1,4146.239304,5458.163525,79969.50521,8,615,0
9997,6694516,2,3088.223727,4813.090925,38192.67591,5,596,0
9998,3942961,0,3288.901666,1043.099660,50929.37206,2,647,0


In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
# Preprocessing pipeline
numeric_features = ['credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)])

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

Model Engineering

In [ ]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.2/738.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72

In [ ]:
# Import des bibliothèques importantes à mon exercice
import mlflow
import mlflow.sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
# Fonction pour train et log
def train_and_log(model, model_name, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=f"{model_name}_run"):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred),
            'auc_roc': roc_auc_score(y_test, y_prob)
        }

        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")
        print(f"Logged {model_name} with AUC-ROC: {metrics['auc_roc']}")

In [ ]:
# Experiments (un par modèle)
mlflow.set_experiment("Decision_Tree")
dt = DecisionTreeClassifier(max_depth=5)
train_and_log(dt, "DT_Base", X_train_preprocessed, y_train, X_test_preprocessed, y_test)

mlflow.set_experiment("Logistic_Regression")
lr = LogisticRegression()
train_and_log(lr, "LR_Base", X_train_preprocessed, y_train, X_test_preprocessed, y_test)

mlflow.set_experiment("Random_Forest")
rf = RandomForestClassifier(n_estimators=100)
train_and_log(rf, "RF_Base", X_train_preprocessed, y_train, X_test_preprocessed, y_test)

# Itérations : e.g., pour RF, ajoute un run avec tuning
with mlflow.start_run(run_name="RF_Tuned"):
    rf_tuned = RandomForestClassifier(n_estimators=200, max_depth=10)
    # ... train et log comme ci-dessus

2025/10/15 20:51:14 INFO mlflow.tracking.fluent: Experiment with name 'Decision_Tree' does not exist. Creating a new experiment.
2025/10/15 20:51:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/15 20:51:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/15 20:51:27 INFO mlflow.tracking.fluent: Experiment with name 'Logistic_Regression' does not exist. Creating a new experiment.
2025/10/15 20:51:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged DT_Base with AUC-ROC: 0.9994055712153872


2025/10/15 20:51:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/15 20:51:31 INFO mlflow.tracking.fluent: Experiment with name 'Random_Forest' does not exist. Creating a new experiment.


Logged LR_Base with AUC-ROC: 0.9999883933012768


2025/10/15 20:51:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/15 20:51:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged RF_Base with AUC-ROC: 0.9998831039628586


Déploiement du meilleur modèle

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.3 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import joblib
import numpy as np

# Load model et preprocessor (sauvegardés via joblib.dump)
model = joblib.load('best_model.pkl')
preprocessor = joblib.load('preprocessor.pkl')

st.title("Prédiction de Défaut de Prêt")

# Inputs
age = st.number_input("Âge", min_value=18)
income = st.number_input("Revenus annuels")
# Ajoute d'autres features...

if st.button("Prédire"):
    input_data = np.array([[age, income, ...]])  # Adapter
    input_preprocessed = preprocessor.transform(input_data)
    prob = model.predict_proba(input_preprocessed)[0][1]
    st.write(f"Probabilité de défaut : {prob:.2f}")

FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pkl'

In [ ]:
import mlflow
import joblib
import pandas as pd

# Find the best run based on AUC-ROC
best_run = mlflow.search_runs(order_by=["metrics.auc_roc DESC"], max_results=1).iloc[0]
best_run_id = best_run.run_id
best_model_uri = f"runs:/{best_run_id}/model"

# Load the best model
best_model = mlflow.sklearn.load_model(best_model_uri)

# Save the best model and the preprocessor
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')

print(f"Best model (Run ID: {best_run_id}) and preprocessor saved successfully.")

Best model (Run ID: cd2064d9dc1c4669afee37d5a7df1c54) and preprocessor saved successfully.


Push dans github

In [19]:
# Initialize a Git repository, add files, and commit
!git init
!git add .
!git commit -m "Initial commit with data loading, preprocessing, model training, and best model saving"

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@90dd4b5e9954.(none)')


In [20]:
# Config

!git config --global user.name "ccheikh-ismael"
!git config --global user.email "ccheikh.ismaelpro@gmail.com"

In [21]:
# Clonage du dépôt

!git clone https://github.com/BINTOUDIOP/Projet_MLOPS.git

Cloning into 'Projet_MLOPS'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 253 (delta 30), reused 202 (delta 27), pack-reused 45 (from 2)
Receiving objects: 100% (253/253), 1.30 MiB | 9.50 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [22]:
# Aller dans le dossier du dépôt
%cd Projet_MLOPS

/content/Projet_MLOPS


In [23]:
# branche-cic

!git fetch
!git checkout -b branche-cic origin/branche-cic

Branch 'branche-cic' set up to track remote branch 'branche-cic' from 'origin'.
Switched to a new branch 'branche-cic'


In [28]:
!cp /content/Projet_MLOPS.ipynb /content/Projet_MLOPS/

cp: cannot stat '/content/Projet_MLOPS.ipynb': No such file or directory


In [26]:
from google.colab import files

In [27]:
!jupyter nbconvert --to notebook --output="/content/mon_notebook.ipynb" "/content/Projet_MLOPS.ipynb"

[NbConvertApp] WARNING | pattern '/content/Projet_MLOPS.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--e

In [29]:
!cp /content/mon_notebook.ipynb /content/Projet_MLOPS/


cp: cannot stat '/content/mon_notebook.ipynb': No such file or directory
